# Pattern 8: Three-Legged OAuth

The agent uses a token obtained through the user's browser via the authorization code flow. The user explicitly consented in a browser redirect. The agent never sees the user's password.

**What the service sees:** A consent-obtained JWT. The agent is fully out of the credential chain.

This is the gold standard for delegated agent authorization.

In [ ]:
from framework.runner import PatternRunner
runner = PatternRunner("p08_three_legged_oauth")

## The auth code

This is the lesson. Read both files to understand what happens at each boundary.

In [ ]:
runner.show_auth_code()

The MCP server uses a pre-obtained consent token. The user went through Keycloak's browser-based consent flow to obtain it. The agent never touched the user's password.

## Consent flow

Before starting the pattern, we need to obtain a token through the browser.

**Steps:**
1. Set `CONSENT_USER` below to the user you will log in as (alice, bob, or dave)
2. Run the consent flow cell -- it will print a URL
3. Open the URL in your browser and log in as that user (password: `password`)
4. The browser will redirect to a dead URL -- that's expected
5. Copy the `code=<...>` value from the URL bar and paste it when prompted

The username you set here **must** match the user you log in as in the browser.
The JWT inside the consent token identifies the user, and the service will
filter data based on that identity.

In [ ]:
# Set this to the user you will log in as in the browser
CONSENT_USER = "dave"

In [ ]:
# Inject the consent token into the running MCP auth handler
import sys
mod = sys.modules.get("_pattern_p08_three_legged_oauth_mcp_auth")
if mod:
    mod.auth_handler.access_token = access_token
    print(f"Consent token set for {CONSENT_USER}")
else:
    print("ERROR: MCP auth module not loaded -- did you run start()?")

In [ ]:
## Run scenarios

The agent will use the consent token for all requests. The username passed
to `run_as` must match `CONSENT_USER` -- the JWT inside the token identifies
the user, and the service filters data based on that proven identity.

In [ ]:
await runner.run_as(CONSENT_USER, "What are my expenses?")

await runner.run_as(CONSENT_USER, "Search for documents")

In [ ]:
await runner.run_as("dave", "What are my expenses?")

In [ ]:
await runner.run_as("dave", "Search for documents")

## What did the service see?

The punchline: what identity did the backend service actually extract?

In [ ]:
await runner.show_service_identity()

The service sees a validated JWT, just like patterns 5-7. The difference is invisible on the wire: this token was obtained through user consent, not through the agent holding the user's password.

The agent is completely out of the credential chain. It never saw a password. It cannot fabricate tokens for other users. It can only act within the scope the user consented to.

In production, the consent screen would typically request specific OAuth2 scopes (e.g. `expenses:read` but not `expenses:approve`), giving the user fine-grained control over what the agent can do. The service would then enforce those scopes alongside the identity claims. We don't implement scope restrictions in this repo (the mechanics are the same as claim-based checks -- the service reads a token field and enforces it), but scopes are a natural complement to the consent flow shown here.

This completes the eight-pattern progression from no identity to fully delegated, consent-based authorization.

In [ ]:
await runner.stop()